# 02 - Feature Engineering
Build model-ready features and forward tracking error targets.

In [ ]:
from pathlib import Path
import importlib.util
import subprocess
import sys

project_root = Path.cwd().resolve()
if project_root.name == "notebooks":
    project_root = project_root.parent

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

for package in ["yfinance", "plotly"]:
    if importlib.util.find_spec(package) is None:
        subprocess.check_call([sys.executable, "-m", "pip", "install", package])

import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from config import PAIR_CONFIGS
from src.data_loader import MarketDataLoader
from src.features import FeatureEngineer

loader  = MarketDataLoader()
panel   = loader.fetch_universe(PAIR_CONFIGS, period="2y", interval="1d")
engineer = FeatureEngineer(rolling_window=20, horizon=1)
features = engineer.transform_universe(panel)

print(f"Feature panel : {features.shape[0]:,} rows × {features.shape[1]} columns")
print(f"Pairs present : {features['pair'].unique().tolist()}")
features.head()

,etf_close,etf_high,etf_low,etf_open,etf_volume,benchmark_close,benchmark_high,benchmark_low,benchmark_open,benchmark_volume,...,benchmark_vol_60,spread_mean_60,spread_std_60,rolling_beta,price_ratio,ratio_zscore,volume_change_1,hl_spread,realized_te,target_te
Date,,,,,,,,,,,,,,,,,,,,,
2024-04-10 00:00:00+00:00,48.486687,48.732095,48.222404,48.297912,1897900,5000.830078,5037.180176,4951.189941,5006.500000,28802000,...,NaN,NaN,NaN,NaN,0.009696,NaN,NaN,0.010512,NaN,NaN
2024-04-10 00:00:00+00:00,501.905121,503.896631,499.923382,501.280312,82652800,5160.640137,5178.430176,5138.700195,5167.879883,3845930000,...,NaN,NaN,NaN,NaN,0.097256,NaN,NaN,0.007916,NaN,NaN
2024-04-10 00:00:00+00:00,137.942810,138.534382,137.525793,137.913718,161300,104.793976,105.209787,104.378165,104.706942,7321600,...,NaN,NaN,NaN,NaN,1.316324,NaN,NaN,0.007312,NaN,NaN
2024-04-10 00:00:00+00:00,433.607971,434.468516,431.540679,432.252858,61502200,18011.660156,18040.830078,17932.419922,17957.960938,5308250000,...,NaN,NaN,NaN,NaN,0.024074,NaN,NaN,0.006752,NaN,NaN
2024-04-11 00:00:00+00:00,48.467819,48.543331,47.769356,48.543331,1977400,4966.680176,5010.089844,4934.399902,4994.149902,30613200,...,NaN,NaN,NaN,NaN,0.009759,NaN,0.041888,0.015969,NaN,NaN


In [ ]:
# --- Target coverage and feature completeness per pair -------------------------
summary = (
    features.groupby("pair")
    .agg(
        total_rows     =("target_te", "count"),
        labelled_rows  =("target_te", lambda s: s.notna().sum()),
        feature_na_pct =("realized_te", lambda s: s.isna().mean() * 100),
        mean_target_te =("target_te", "mean"),
        std_target_te  =("target_te", "std"),
    )
    .round(6)
    .reset_index()
)
summary

pair
FEZ_^STOXX50E    464
QQQ_^NDX         482
SPY_^GSPC        482
URTH_ACWI        482
dtype: int64

In [ ]:
# --- Realized tracking error over time per pair --------------------------------
te_data = features.dropna(subset=["realized_te"]).copy()

fig = go.Figure()
for pair_name, pair_df in te_data.groupby("pair"):
    pair_df = pair_df.sort_index()
    fig.add_trace(go.Scatter(
        x=pair_df.index,
        y=pair_df["realized_te"] * 100,
        mode="lines",
        name=pair_name,
    ))

fig.update_layout(
    title="Rolling Realized Tracking Error Over Time (20-day window, %)",
    xaxis_title="Date",
    yaxis_title="Realized TE (%)",
    template="plotly_white",
    height=430,
    legend=dict(orientation="h", y=1.05, x=0),
)
fig.show()

In [ ]:
# --- Feature correlation with the target variable ------------------------------
# High absolute correlation → feature is a strong linear TE predictor.
# This is exploratory; the model captures non-linear structure beyond this.
target_col = "target_te"
numeric_features = [c for c in features.select_dtypes("number").columns
                    if c not in {target_col, "etf_volume", "benchmark_close", "etf_close",
                                 "etf_open", "etf_high", "etf_low",
                                 "benchmark_open", "benchmark_high", "benchmark_low"}]

corr = (
    features[numeric_features + [target_col]]
    .dropna()
    .corr()[target_col]
    .drop(target_col)
    .sort_values(key=abs, ascending=False)
    .head(20)
)

fig2 = px.bar(
    x=corr.values,
    y=corr.index,
    orientation="h",
    title="Top 20 Features by Linear Correlation with Target TE",
    labels={"x": "Pearson r", "y": "Feature"},
    color=corr.values,
    color_continuous_scale=["#cf2e2e", "#f0f0f0", "#1d8a4c"],
    color_continuous_midpoint=0,
    template="plotly_white",
    height=520,
)
fig2.update_layout(coloraxis_showscale=False, yaxis=dict(autorange="reversed"))
fig2.show()